# Moving application of ShellSIM from 1D time series data to complex 4D gridded dataset

Needed Variables 

T_timeseries=  Temperature Time series  
S_timeseries=  Practical Salinity Time series  
Chl_timeseries= Chlorophyll Time series  
POC_timeseries= Particulate Organic Carbon Time series  
POM_timeseries=  Particulate Organic Matter Time series  
TPM_timeseries= Total Particulate Matter   Time series  

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import pyfabm
import os
from dask.diagnostics import ProgressBar
import warnings
import gc

from dask.distributed import Client, LocalCluster
import dask
import datetime
from contextlib import redirect_stdout,  redirect_stderr # For redirecting stdout, and stderr
import io
import psutil


In [2]:
# Define the integration time period 
start = '02-06-2021'
end = '04-06-2021'
time_horizon = pd.date_range(start=start, end=end, freq='1d')
time_horizon_len = len(time_horizon)

In [3]:
#  Lat lon chunking method
chunking_config={'time': -1, 'latitude': 80, 'longitude': 110}

def load_nc_file(file_path, var_name_in_file, chunking_config=chunking_config):
    """
    Loads a NetCDF file or creates a fake one if not found.
    Args:
        file_path (str): Path to the .nc file.
        var_name_in_file (str): The name of the variable to use.
        chunking_config (dict): Dask chunking configuration.
    Returns:
        xr.Dataset
    """
    fake_filename = f'{var_name_in_file}_gridded_FAKE.nc'
    if os.path.exists(file_path):
        print(f"Successfully loaded: {var_name_in_file}\n")
        return xr.open_dataset(file_path, chunks=chunking_config)

    if os.path.exists(fake_filename):
        print(f"Using existing fake dataset: {fake_filename}\n")
        return xr.open_dataset(fake_filename, chunks=chunking_config)

    print(f"Creating fake gridded data for variable '{var_name_in_file}'...")
    fake_data = np.random.rand(time_horizon_len, 100, 100)
    fake_lats = np.linspace(40, 50, 100)
    fake_lons = np.linspace(-10, 0, 100)

    fake_gridded_dataset = xr.Dataset(
        {var_name_in_file: (['time', 'latitude', 'longitude'], fake_data)},
        coords={'time': time_horizon, 'latitude': fake_lats, 'longitude': fake_lons}
    )
    fake_gridded_dataset.to_netcdf(fake_filename)
    print(f"Saved fake dataset to: {fake_filename}\n")
    return xr.open_dataset(fake_filename, chunks=chunking_config)
        

In [4]:
# Define datasets to be read and used for model
poc_file_path="/home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_BIO_BGC_3D_REP_015_010/cmems_obs-mob_glo_bgc-chl-poc_my_0.25deg_P7D-m_poc_13.00W-42.00E_30.00N-70.00N_0.00-1000.00m_2021-06-01-2021-06-30_3f1f32d8adbe3d686e11d7d4a40be7bb.nc"
poc_var_name = 'poc'  # variable name inside poc.nc

salinity_file_path="/home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc"
salinity_var_name = 'sos' # variable name inside salinity.nc

# Fake data generation to be triggered in the 'except' block of load_nc_file if path doesn't exist
temp_file_path = 'non_existent_temp.nc'
temp_var_name = 'temperature' # variable name inside temperature.nc

chl_file_path = 'non_existent_chl.nc'
chl_var_name = 'chl'

pom_file_path = 'non_existent_pom.nc'
pom_var_name = 'pom'

tpm_file_path = 'non_existent_tpm.nc'
tpm_var_name = 'tpm'


In [5]:
# Load all datasets
print("Loading datasets...\n")
ds_poc = load_nc_file(poc_file_path, poc_var_name)
ds_sal = load_nc_file(salinity_file_path, salinity_var_name)
ds_temp = load_nc_file(temp_file_path, temp_var_name)
ds_chl = load_nc_file(chl_file_path, chl_var_name)
ds_pom = load_nc_file(pom_file_path, pom_var_name)
ds_tpm = load_nc_file(tpm_file_path, tpm_var_name)

Loading datasets...

Successfully loaded: poc



getfattr: /home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_BIO_BGC_3D_REP_015_010/cmems_obs-mob_glo_bgc-chl-poc_my_0.25deg_P7D-m_poc_13.00W-42.00E_30.00N-70.00N_0.00-1000.00m_2021-06-01-2021-06-30_3f1f32d8adbe3d686e11d7d4a40be7bb.nc: Operation not supported


Successfully loaded: sos



getfattr: /home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc: Operation not supported


Using existing fake dataset: temperature_gridded_FAKE.nc

Using existing fake dataset: chl_gridded_FAKE.nc

Using existing fake dataset: pom_gridded_FAKE.nc

Using existing fake dataset: tpm_gridded_FAKE.nc



In [6]:
# Align coordinates - using POC data as reference
# Grid alignment using ds_poc as the reference grid and interpolating all other variables to it, 
# as a way to handle mismatched grids.
print("Aligning coordinates...")
ref_lats = ds_poc.latitude
ref_lons = ds_poc.longitude


Aligning coordinates...


In [7]:
interp_kwargs = {'fill_value': 'extrapolate'}

# For 4D variables: Select first depth values -  Select a single depth level and interpolate 
# use .sel(depth=0, method='nearest') to grab the surface layer
poc_daily = ds_poc[poc_var_name].sel(depth=0, method='nearest').interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
sal_daily = ds_sal[salinity_var_name].sel(depth=0, method='nearest').interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)


# Process 3D variables (datasets with no depth)
temp_daily = ds_temp[temp_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
chl_daily = ds_chl[chl_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
pom_daily = ds_pom[pom_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
tpm_daily = ds_tpm[tpm_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)


# Merge into a single dataset
ds_daily = xr.Dataset({
    'salinity': sal_daily,
    'POC': poc_daily,
    'temperature': temp_daily,
    'Chl': chl_daily,
    'POM': pom_daily,
    'TPM': tpm_daily
})

print("Merged daily dataset")
ds_daily


Merged daily dataset


<xarray.Dataset> Size: 93MB
Dimensions:      (time: 60, latitude: 160, longitude: 220)
Coordinates:
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
  * latitude     (latitude) float32 640B 30.12 30.38 30.62 ... 69.38 69.62 69.88
  * longitude    (longitude) float32 880B -12.88 -12.62 -12.38 ... 41.62 41.88
    depth        float32 4B 0.0
Data variables:
    salinity     (time, latitude, longitude) float64 17MB dask.array<chunksize=(10, 54, 220), meta=np.ndarray>
    POC          (time, latitude, longitude) float32 8MB dask.array<chunksize=(15, 80, 220), meta=np.ndarray>
    temperature  (time, latitude, longitude) float64 17MB dask.array<chunksize=(15, 80, 220), meta=np.ndarray>
    Chl          (time, latitude, longitude) float64 17MB dask.array<chunksize=(15, 80, 220), meta=np.ndarray>
    POM          (time, latitude, longitude) float64 17MB dask.array<chunksize=(15, 80, 220), meta=np.ndarray>
    TPM          (time, latitude, longitude) float64 17MB dask.array<chunksize=(15, 80, 220), meta=np.ndarray>

# ShellSIM model wraapper   
takes a 1D numpy array (time-series for one pixel) as input. run entire for loop (time-stepping) and return a 1D numpy array of the result ( eg soft tissue energy time-series). Then Apply in Parallel, using xr.apply_ufunc to apply wrapper function to the gridded data. tell apply_ufunc that the "core dimension" is time, which instructs it to parallelize over all other dimensions (lat, lon)

In [8]:

# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# RUN_LOG_FILENAME = f"fabm_run_log_{timestamp}.log"

def run_fabm_at_point_full(T_ts, S_ts, Chl_ts, POC_ts, POM_ts, TPM_ts):
    """
    Runs FABM time-loop for a single spatial point.
    Returns a 1D array of output, e.g., Soft Tissue Energy.
    """

    # Convert inputs to float64 to prevent overflow
    T_ts = np.asarray(T_ts, dtype=np.float64)
    S_ts = np.asarray(S_ts, dtype=np.float64)
    Chl_ts = np.asarray(Chl_ts, dtype=np.float64)
    POC_ts = np.asarray(POC_ts, dtype=np.float64)
    POM_ts = np.asarray(POM_ts, dtype=np.float64)
    TPM_ts = np.asarray(TPM_ts, dtype=np.float64)


    
    # Check for NaN inputs
    if (np.any(np.isnan(T_ts)) or np.any(np.isnan(S_ts)) or 
        np.any(np.isnan(Chl_ts)) or np.any(np.isnan(POC_ts)) or 
        np.any(np.isnan(POM_ts)) or np.any(np.isnan(TPM_ts))):
        return np.full(time_horizon_len, np.nan)
    
    # Check for non-finite values
    if not (np.all(np.isfinite(T_ts)) and np.all(np.isfinite(S_ts)) and
            np.all(np.isfinite(Chl_ts)) and np.all(np.isfinite(POC_ts)) and
            np.all(np.isfinite(POM_ts)) and np.all(np.isfinite(TPM_ts))):
        return np.full(time_horizon_len, np.nan)


    # Create a persistent null output buffer
    null_buffer = io.StringIO()
    
    try:
        # --- START OUTPUT REDIRECTION BLOCK ---
        # Redirect both stdout and stderr to the buffer
        with redirect_stdout(null_buffer), redirect_stderr(null_buffer):
            # Initialize model
            model = pyfabm.Model("/home/jovyan/work/ShellSIM_Trials/notebook_timeseries/fabm.yaml")
            
            # Set static dependencies
            model.cell_thickness = 1.0
            model.dependencies["seeding_rate"].value = 0.0
            model.dependencies["harvest_ratio"].value = 0.0
            model.dependencies["current_speed"].value = 1.0
            model.dependencies["air_exposure"].value = 0.0
            model.dependencies["number_of_days_since_start_of_the_year"].value = 0.0
    
    
            # Set INITIAL values for all dependencies
            # We use the first day's value (index 0) to initialize.
            model.dependencies["temperature"].value = float(T_ts[0])
            model.dependencies["practical_salinity"].value = float(S_ts[0])
            model.findStateVariable('Chl1/Chl').value = float(Chl_ts[0])
            model.findStateVariable('POC1/POC').value = float(POC_ts[0])
            model.findStateVariable('POM1/POM').value = float(POM_ts[0])
            model.findStateVariable('TPM1/TPM').value = float(TPM_ts[0])
    
            if not model.start():
                raise RuntimeError("FABM model failed to start internally.")
    
            # Initialize output
            outputs = np.zeros(time_horizon_len)
        
            # Daily integration
            for nd in range(time_horizon_len):
                # Set environmental variables
                model.dependencies["temperature"].value = float(T_ts[nd])
                model.dependencies["practical_salinity"].value = float(S_ts[nd])
                
                # Set food variables
                model.findStateVariable('Chl1/Chl').value = float(Chl_ts[nd])
                model.findStateVariable('POC1/POC').value = float(POC_ts[nd])
                model.findStateVariable('POM1/POM').value = float(POM_ts[nd])
                model.findStateVariable('TPM1/TPM').value = float(TPM_ts[nd])
                
                # Calculate and apply growth rates
                state_rates = model.getRates()
                model.state[:] += state_rates * 86400.
                
                # Save output (soft_tissue_output)
                outputs[nd] = model.state[0]
    
            return outputs
        
            #      # Save all 11 states
            #     outputs[:, nd] = model.state[:] 
            # # RETURN THE FULL 2D OUTPUT ARRAY
            # return outputs
        
    except RuntimeError as e: # Catch the internal failure we raised
        warnings.warn("!!! Model failed to start even after setting initial dependencies.")
        # We can still print the specific FABM error message outside the redirection
        print(f"Model failed to start: {pyfabm.getError()}")
        return np.full(time_horizon_len, np.nan)
        
    except Exception as e:
        warnings.warn(f"FABM error: {str(e)}")
        return np.full(time_horizon_len, np.nan)

In [9]:
# Rechunk for optimal performance
# After all the merging and interpolating, Dask's chunks can get fragmented, rechunk so every Dask task receives 
# a data chunk of the exact size


print("Rechunking dataset for optimal performance")
ds_daily = ds_daily.chunk({
    'time': -1,        # Keep entire time series together (CRITICAL!)
    'latitude': 10,    # Process 40x40 spatial tiles
    'longitude': 10    # Adjust based on your memory constraints
})
print("Rechunked dataset")
print(ds_daily.chunks)

Rechunking dataset for optimal performance
Rechunked dataset
Frozen({'time': (60,), 'latitude': (10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10), 'longitude': (10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10)})


## Most optimal processing method:  parallelize over the spatial dimensions (latitude, longitude) using xarray.apply_ufunc.  
chunking only the spatial dimensions (latitude and longitude) tells Dask to split the map into tiles, but keep the full time series for each pixel intact.

# Run computation and Save  
Call .compute() or .to_netcdf() on the result. This triggers Dask to execute the parallel computation and write the final 3D output file.

In [10]:
# Optional: Start a local Dask cluster for better monitoring
# Adjust n_workers and threads_per_worker based on your system

# 2025-11-10 23:14:18,367 - distributed.worker.memory - WARNING - 
# Worker is at 80% memory usage. Pausing worker.  Process memory: 2.99 GiB -- Worker memory limit: 3.73 GiB

# Get available memory
available_gb = psutil.virtual_memory().available / (1024**3)
print(f"Available memory: {available_gb:.2f} GB")

# Use only 60% of available memory for Dask
dask_memory = available_gb * 0.6
memory_per_worker = dask_memory / 2  # 2 workers

try:
    client = Client(
        n_workers=2,
        threads_per_worker=1,
        memory_limit=f'{memory_per_worker:.1f}GB',
        silence_logs=True  # Reduce logging overhead
    )
    print(f"Dask dashboard: {client.dashboard_link}")
    print(f"Memory per worker: {memory_per_worker:.2f} GB\n")
    
except:
    print("Using default Dask scheduler")


# Set output filename
output_file_name = "gridded_oyster_output_soft_tissue_energy_optimized.nc"
time_horizon_len = ds_daily.time.size


# Single apply_ufunc call - Dask handles all parallelization
result_sten = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # Input arrays (full dataset, not batched)
    ds_daily['temperature'],   
    ds_daily['salinity'],
    ds_daily['Chl'],
    ds_daily['POC'],
    ds_daily['POM'],
    ds_daily['TPM'],
    
    input_core_dims=[['time']] * 6,
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    dask='parallelized',  # This triggers parallel execution
    vectorize=True, 
    output_dtypes=[float],
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)

# Add metadata
result_sten = result_sten.assign_coords(time=ds_daily.time)
result_sten = result_sten.rename('soft_tissue_energy')
result_sten.attrs['units'] = 'J'
result_sten.attrs['long_name'] = 'Oyster Soft Tissue Energy'

# Convert to Dataset and compute/save in one operation
print("Starting parallel computation...")
with ProgressBar():
    result_ds = result_sten.to_dataset()
    
    # Optional: Add compression to reduce file size
    encoding = {
        'soft_tissue_energy': {
            'zlib': True, 
            'complevel': 4,
            'dtype': 'float32'  # Use float32 instead of float64 to save space
        }
    }
    
    result_ds.to_netcdf(
        output_file_name,
        format='NETCDF4',
        encoding=encoding,
        compute=True
    )

print(f"✅ Success: Results saved to {output_file_name}")

# Optional: Close the client
try:
    client.close()
except:
    pass

    

2026-06-04 11:48:07,864 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-06-04 11:48:07,867 - distributed.scheduler - INFO - State start
2026-06-04 11:48:07,871 - distributed.scheduler - INFO -   Scheduler at:     tcp://127.0.0.1:39529
2026-06-04 11:48:07,871 - distributed.scheduler - INFO -   dashboard at:  http://127.0.0.1:8787/status
2026-06-04 11:48:07,872 - distributed.scheduler - INFO - Registering Worker plugin shuffle
2026-06-04 11:48:07,881 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:41169'


Available memory: 54.25 GB


2026-06-04 11:48:08,287 - distributed.nanny - INFO -         Start Nanny at: 'tcp://127.0.0.1:43959'
2026-06-04 11:48:08,722 - distributed.worker - INFO -       Start worker at:      tcp://127.0.0.1:42297
2026-06-04 11:48:08,722 - distributed.worker - INFO -          Listening to:      tcp://127.0.0.1:42297
2026-06-04 11:48:08,722 - distributed.worker - INFO -           Worker name:                          0
2026-06-04 11:48:08,722 - distributed.worker - INFO -          dashboard at:            127.0.0.1:38147
2026-06-04 11:48:08,722 - distributed.worker - INFO - Waiting to connect to:      tcp://127.0.0.1:39529
2026-06-04 11:48:08,722 - distributed.worker - INFO - -------------------------------------------------
2026-06-04 11:48:08,722 - distributed.worker - INFO -               Threads:                          1
2026-06-04 11:48:08,722 - distributed.worker - INFO -                Memory:                  15.18 GiB
2026-06-04 11:48:08,722 - distributed.worker - INFO -       Local D

Dask dashboard: http://127.0.0.1:8787/status
Memory per worker: 16.28 GB

Starting parallel computation...


getfattr: /home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc: Operation not supported
getfattr: /home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc: Operation not supported
2026-06-04 11:48:10,457 - distributed.worker - ERROR - Compute Failed
Key:       ('interpnd-interpnd_0-5c01197376b908327e6583d21123b2d4', 1, 1, 0)
State:     executing
Task:  <Task ('interpnd-interpnd_0-5c01197376b908327e6583d21123b2d4', 1, 1, 0) _execute_subgraph(...)>
Exception: 'ImportError("/lib/x86_64-linux-gnu/libstdc++.so.6: version `CXXABI_1.3.15\' not found (required by /opt/conda/lib/python3.11/site-packages/scipy/optimize/_highspy/_core.cpython-311-x86_64-linux-gnu.so)")

ImportError: /lib/x86_64-linux-gnu/libstdc++.so.6: version `CXXABI_1.3.15' not found (required by /opt/conda/lib/python3.11/site-packages/scipy/optimize/_highspy/_core.cpython-311-x86_64-linux-gnu.so)

In [ ]:
# # Process half the domain at a time
# lat_mid = len(ds_daily.lat) // 2

# # First half
# ds_part1 = ds_daily.isel(lat=slice(0, lat_mid))
# result_part1 = xr.apply_ufunc(...)  # your apply_ufunc call
# result_part1.to_netcdf("part1.nc")

# # Second half
# ds_part2 = ds_daily.isel(lat=slice(lat_mid, None))
# result_part2 = xr.apply_ufunc(...)
# result_part2.to_netcdf("part2.nc")

# # Merge results
# result_final = xr.concat([result_part1, result_part2], dim='lat')